In [15]:
import openai 
import os
import json
from qdrant_client import QdrantClient
from qdrant_client.models import Filter, FieldCondition, MatchValue
from langsmith import Client

### Download all data from Qdrant

In [42]:
qdrant_client = QdrantClient(url =  "http://localhost:6333")

In [43]:
collection_name="Amazon-shopping-collection-01"
all_points = qdrant_client.scroll(
    collection_name=collection_name,
    limit=50,
    offset=None,
    with_payload=True,
    with_vectors=False
)


In [18]:
all_points

([Record(id=0, payload={'preprocessed_description': 'Beats by Dr. Dre - Beats Solo3 Wireless Headphones - Gold(Renewed)DRIVEN BY THE APPLE W1 CHIP: Incorporating the efficient W1 chip brings seamless setup and switching for your Apple devices, up to 40 hours of battery life, and Fast Fuel technology for 3 hours of play with a 5-minute charge FEEL YOUR MUSIC: At the heart of Beats Solo3 Wireless is award-winning Beats sound. This headphone delivers premium playback with fine-tuned acoustics that maximize clarity, breadth, and balance ALL-DAY PLAY: Beats Solo3 Wireless delivers up to 40 hours of battery life driven by the efficiency of the Apple W1 chip. Integrated on-ear controls coupled with dual beam-forming mics allow you to take calls, play music, adjust volume, and activate Siri while on the go CUSTOMISED COMFORT: The on-ear, cushioned ear cups are adjustable, so you can customize your fit for all-day listening comfort. The comfort-cushion ear cups buffer outside noise for immersiv

In [5]:
all_points[0]

[Record(id=0, payload={'preprocessed_description': 'Beats by Dr. Dre - Beats Solo3 Wireless Headphones - Gold(Renewed)DRIVEN BY THE APPLE W1 CHIP: Incorporating the efficient W1 chip brings seamless setup and switching for your Apple devices, up to 40 hours of battery life, and Fast Fuel technology for 3 hours of play with a 5-minute charge FEEL YOUR MUSIC: At the heart of Beats Solo3 Wireless is award-winning Beats sound. This headphone delivers premium playback with fine-tuned acoustics that maximize clarity, breadth, and balance ALL-DAY PLAY: Beats Solo3 Wireless delivers up to 40 hours of battery life driven by the efficiency of the Apple W1 chip. Integrated on-ear controls coupled with dual beam-forming mics allow you to take calls, play music, adjust volume, and activate Siri while on the go CUSTOMISED COMFORT: The on-ear, cushioned ear cups are adjustable, so you can customize your fit for all-day listening comfort. The comfort-cushion ear cups buffer outside noise for immersive

In [23]:
all_contxt = [ {"id": data.payload["parent_asin"], "text": data.payload["preprocessed_description"]} for data in all_points[0]]
all_contxt

[{'id': 'B075NR47FN',
  'text': 'Beats by Dr. Dre - Beats Solo3 Wireless Headphones - Gold(Renewed)DRIVEN BY THE APPLE W1 CHIP: Incorporating the efficient W1 chip brings seamless setup and switching for your Apple devices, up to 40 hours of battery life, and Fast Fuel technology for 3 hours of play with a 5-minute charge FEEL YOUR MUSIC: At the heart of Beats Solo3 Wireless is award-winning Beats sound. This headphone delivers premium playback with fine-tuned acoustics that maximize clarity, breadth, and balance ALL-DAY PLAY: Beats Solo3 Wireless delivers up to 40 hours of battery life driven by the efficiency of the Apple W1 chip. Integrated on-ear controls coupled with dual beam-forming mics allow you to take calls, play music, adjust volume, and activate Siri while on the go CUSTOMISED COMFORT: The on-ear, cushioned ear cups are adjustable, so you can customize your fit for all-day listening comfort. The comfort-cushion ear cups buffer outside noise for immersive sound, so you can 

### Build a prompt to generate synthetic eval reference dataset

In [20]:
output_schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "reasoning": {
                "type": "string",
                "description": "Reasoning why the question could be answered with the chunks.",
            },
            "question": {
                "type": "string",
                "description": "Suggested question.",
            },
            "chunk_ids": {
                "type": "array",
                "items": {
                    "type": "string",
                    "description": "ID of the chunk that could be used to answer the question.",
                },
            },
            "answer_example": {
                "type": "string",
                "description": "Suggested answer grounded in the context.",
            }
        },
    },
}

SYSTEM_PROMPT = f"""
I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, refer to the chunks as products.

<OUTPUT JSON SCHEMA>
{json.dumps(output_schema, indent=2)}
</OUTPUT JSON SCHEMA>

I need to be able to parse the json output.
"""

In [21]:
print(SYSTEM_PROMPT)


I am building a RAG application. I have a collection of 50 chunks of text.
The RAG application will act as a shopping assistant that can answer questions about the stock of the products we have available.
I will provide all of the available products to you with IDs of each chunk.
Come up with 30 questions to which the answers could be grounded in the chunk context.
The questions should imitate a potential real user of this RAG system - a customer of the e-shop.
As an output I need you to provide me the list of questions and the IDs of the chunks that could be used to answer them.
Also, provide an example answer to the question given the context of the chunks.
Also, provide the reason why you chose the chunks to answer the questions.
Construct 10 questions that could use multiple chunks in the answer.
Construct 15 questions that could use single chunk in the answer.
Construct 5 questions that can't be answered with the available chunks.
Don't use word "chunks" in suggested questions, r

In [24]:
USER_PROMPT = f"""
Here is the list of chunks, each list element is a dictionary with id and text:
{json.dumps(all_contxt, indent=2)}
"""

In [25]:
print(USER_PROMPT)


Here is the list of chunks, each list element is a dictionary with id and text:
[
  {
    "id": "B075NR47FN",
    "text": "Beats by Dr. Dre - Beats Solo3 Wireless Headphones - Gold(Renewed)DRIVEN BY THE APPLE W1 CHIP: Incorporating the efficient W1 chip brings seamless setup and switching for your Apple devices, up to 40 hours of battery life, and Fast Fuel technology for 3 hours of play with a 5-minute charge FEEL YOUR MUSIC: At the heart of Beats Solo3 Wireless is award-winning Beats sound. This headphone delivers premium playback with fine-tuned acoustics that maximize clarity, breadth, and balance ALL-DAY PLAY: Beats Solo3 Wireless delivers up to 40 hours of battery life driven by the efficiency of the Apple W1 chip. Integrated on-ear controls coupled with dual beam-forming mics allow you to take calls, play music, adjust volume, and activate Siri while on the go CUSTOMISED COMFORT: The on-ear, cushioned ear cups are adjustable, so you can customize your fit for all-day listening 

In [34]:
response = openai.chat.completions.create(
    model="gpt-5.4",
    messages=[
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": USER_PROMPT}
    ],
    reasoning_effort="none"
)

print(response.choices[0].message.content)

[
  {
    "reasoning": "This can be answered from one product because the Beats Solo3 listing explicitly states battery life and fast-charging details.",
    "question": "How long does the Beats Solo3 battery last, and is there any quick-charge feature?",
    "chunk_ids": ["B075NR47FN"],
    "answer_example": "The Beats Solo3 Wireless offers up to 40 hours of battery life. It also includes Fast Fuel charging, which gives about 3 hours of playback from a 5-minute charge."
  },
  {
    "reasoning": "This is grounded in a single product because the camera backpack description lists its storage capacity and supported items.",
    "question": "Will this camera backpack fit a DSLR, a couple of lenses, a tripod, and a 14-inch laptop?",
    "chunk_ids": ["B071J4M3TF"],
    "answer_example": "Yes. The CADeN camera backpack is described as fitting 1 camera, 2 lenses, a tripod, a 14-inch laptop, and even some clothes."
  },
  {
    "reasoning": "This can be answered from one product because the S

In [27]:
response.usage.completion_tokens

3069

In [38]:
json_output = response.choices[0].message.content
json_output = json_output.replace("```json","").replace("```","")
json_output = json.loads(json_output)
json_output


[{'reasoning': 'This can be answered from one product because the Beats Solo3 listing explicitly states battery life and fast-charging details.',
  'question': 'How long does the Beats Solo3 battery last, and is there any quick-charge feature?',
  'chunk_ids': ['B075NR47FN'],
  'answer_example': 'The Beats Solo3 Wireless offers up to 40 hours of battery life. It also includes Fast Fuel charging, which gives about 3 hours of playback from a 5-minute charge.'},
 {'reasoning': 'This is grounded in a single product because the camera backpack description lists its storage capacity and supported items.',
  'question': 'Will this camera backpack fit a DSLR, a couple of lenses, a tripod, and a 14-inch laptop?',
  'chunk_ids': ['B071J4M3TF'],
  'answer_example': 'Yes. The CADeN camera backpack is described as fitting 1 camera, 2 lenses, a tripod, a 14-inch laptop, and even some clothes.'},
 {'reasoning': 'This can be answered from one product because the Samsung Tab S3 cover text says it uses 

In [39]:
single_chunk_counter = 0
multiple_chunk_counter = 0
impossible_counter = 0

for item in json_output:
    if len(item["chunk_ids"]) == 1:
        single_chunk_counter += 1
    elif len(item["chunk_ids"]) > 1:
        multiple_chunk_counter += 1
    else:
        impossible_counter += 1

In [40]:
print(f"Single chunk questions: {single_chunk_counter}")
print(f"Multiple chunk questions: {multiple_chunk_counter}")
print(f"Impossible questions: {impossible_counter}")

Single chunk questions: 15
Multiple chunk questions: 12
Impossible questions: 5


In [46]:
points = qdrant_client.scroll(
    collection_name=collection_name,
    scroll_filter=Filter(
        must=[
            FieldCondition(
                key="parent_asin",
                match=MatchValue(value="B09KH916K5")
            )
        ]
    )
)[0]

In [47]:
points[0].payload

{'preprocessed_description': '12x42 Compact Binoculars with Low Night Vision for Adults Kids Waterproof BAK4 Prism FMC Lens Binoculars for Hunting Bird Wildlife Watching Travel Sports✨【High Power Binoculars】 - 12x optics magnification binoculars with 42mm big objective lens, large field of view and long distance, see farther and wider. Perfect binoculars for courtyard birding, hunting, hiking, camping, climbing, safari, sports games and concerts. This is ideal binoculars for adults and teens. ✨【Professional Binoculars】 - Professional FMC Multi-Coated Lens, light transmittance up to 98.5%. Reduce chromatic aberration and Images blurry, provide crystal clear and sharp images. Professional BAK-4 prism(BAK 4 has superior reflective and dispersion qualities compared to BAK 7) ensure higher refractive index, more transparent imaging, giving you full HD VR vision experience. ✨【Perfect Eyepiece Size】 - 20MM eyepiece size is perfectly designed for eye structure of American, which provides a bro

In [48]:
def get_description(parent_asin: str) -> str:
    
    points = qdrant_client.scroll(
        collection_name=collection_name,
        scroll_filter=Filter(
            must=[
                FieldCondition(
                    key="parent_asin",
                    match=MatchValue(value=parent_asin)
                )
            ]
        )
    )[0]

    return points[0].payload["preprocessed_description"]

In [49]:
get_description("B09KH916K5")

'12x42 Compact Binoculars with Low Night Vision for Adults Kids Waterproof BAK4 Prism FMC Lens Binoculars for Hunting Bird Wildlife Watching Travel Sports✨【High Power Binoculars】 - 12x optics magnification binoculars with 42mm big objective lens, large field of view and long distance, see farther and wider. Perfect binoculars for courtyard birding, hunting, hiking, camping, climbing, safari, sports games and concerts. This is ideal binoculars for adults and teens. ✨【Professional Binoculars】 - Professional FMC Multi-Coated Lens, light transmittance up to 98.5%. Reduce chromatic aberration and Images blurry, provide crystal clear and sharp images. Professional BAK-4 prism(BAK 4 has superior reflective and dispersion qualities compared to BAK 7) ensure higher refractive index, more transparent imaging, giving you full HD VR vision experience. ✨【Perfect Eyepiece Size】 - 20MM eyepiece size is perfectly designed for eye structure of American, which provides a broader and clearer images for b

### Create Eval dataset in LangSmith

In [50]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [51]:
ls_client = Client()

In [52]:
dataset_name = "Amazon-shopping-collection-01-dataset"
dataset = ls_client.create_dataset(
    dataset_name=dataset_name,
    description="Dataset for Amazon-shopping-collection-01"
)

In [53]:
for item in json_output:
    ls_client.create_example(
        dataset_id=dataset.id,
        inputs={
            "question": item["question"]
        },
        outputs={
            "ground_truth": item["answer_example"],
            "reference_context_ids": item["chunk_ids"],
            "reference_descriptions": [get_description(id) for id in item["chunk_ids"]]
        }
    )